In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import h5py
import numpy as np
import cv2
from tqdm import tqdm
from glob import glob
import random
import shutil
import matplotlib.pyplot as plt

In [3]:
input_dir = "dataset/data"
output_image_dir = "yolo/images"
original_mask_dir = "yolo/masks"
highlight_before_dir = "yolo/highlighted_overlay_before"
highlight_after_dir = "yolo/highlighted_overlay_after"
output_label_dir = "yolo/labels"

# Create folders
for d in [output_image_dir, original_mask_dir, highlight_before_dir, highlight_after_dir, output_label_dir]:
    os.makedirs(d, exist_ok=True)
    
# Morph kernel for alignment
kernel = np.ones((3, 3), np.uint8)

## Convert .mat to images + original masks

In [4]:
for mat_file in tqdm(sorted(os.listdir(input_dir))):
    if not mat_file.endswith(".mat"):
        continue

    mat_path = os.path.join(input_dir, mat_file)
    with h5py.File(mat_path, "r") as f:
        image = np.array(f["cjdata/image"]).T
        mask = np.array(f["cjdata/tumorMask"]).T
        label_value = int(f["cjdata/label"][0][0])  # 1,2,3

    # Normalize image if needed
    if image.max() > 255:
        image = (image / image.max()) * 255
    image = image.astype(np.uint8)
    mask = (mask > 0).astype(np.uint8) * 255

    # Aligned mask (eroded)
    aligned_mask = cv2.erode(mask, kernel, iterations=1)

    # File name
    fname = mat_file.replace(".mat", ".png")

    # Save original image and mask
    cv2.imwrite(os.path.join(output_image_dir, fname), image)
    cv2.imwrite(os.path.join(original_mask_dir, fname), mask)

    # Convert image to BGR for overlay
    image_bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    overlay_b = image_bgr.copy()
    overlay_a = image_bgr.copy()

    # Contours
    contours_b, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_a, _ = cv2.findContours(aligned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Draw red contours
    cv2.drawContours(overlay_b, contours_b, -1, (0, 0, 255), 2)
    cv2.drawContours(overlay_a, contours_a, -1, (0, 0, 255), 2)

    # Save highlighted overlays
    cv2.imwrite(os.path.join(highlight_before_dir, fname), overlay_b)
    cv2.imwrite(os.path.join(highlight_after_dir, fname), overlay_a)

    # Create YOLO labels
    h, w = mask.shape
    annotations = []
    for contour in contours_a:  # use aligned mask for labeling
        x, y, bw, bh = cv2.boundingRect(contour)
        x_center = (x + bw / 2) / w
        y_center = (y + bh / 2) / h
        bw_norm = bw / w
        bh_norm = bh / h
        class_id = label_value - 1  # convert to 0-based class
        annotations.append(f"{class_id} {x_center} {y_center} {bw_norm} {bh_norm}")

    label_path = os.path.join(output_label_dir, fname.replace(".png", ".txt"))
    with open(label_path, "w") as f:
        f.write("\n".join(annotations))

100%|██████████████████████████████████████████████████████████████████████████████| 3064/3064 [01:30<00:00, 33.91it/s]


# Train/Val/Test split

In [6]:
images = sorted(os.listdir(output_image_dir))
random.seed(42)
random.shuffle(images)

n_total = len(images)
n_train = int(n_total * 0.7)
n_val = int(n_total * 0.2)
n_test = n_total - n_train - n_val

train_files = images[:n_train]
val_files = images[n_train:n_train + n_val]
test_files = images[n_train + n_val:]

splits = {"train": train_files, "val": val_files, "test": test_files}
for split in splits:
    os.makedirs(f"yolo/{split}/images", exist_ok=True)
    os.makedirs(f"yolo/{split}/labels", exist_ok=True)

def copy_split(files, split):
    for fname in files:
        img_path = os.path.join(output_image_dir, fname)
        label_path = os.path.join(output_label_dir, fname.replace(".png", ".txt"))
        shutil.copy(img_path, f"yolo/{split}/images/{fname}")
        shutil.copy(label_path, f"yolo/{split}/labels/{fname.replace('.png', '.txt')}")

for split, files in splits.items():
    copy_split(files, split)

print(f"Splitting complete: Train={len(train_files)} Val={len(val_files)} Test={len(test_files)}")


Splitting complete: Train=2144 Val=612 Test=308
